In [ ]:
# ==============================================================================
# PIPELINE FILOGENÉTICO - TOPOLOGÍA NATIVA EXACTA (RESPETA RAÍZ ORIGINAL)
# ==============================================================================
!pip install biopython matplotlib -q

import io
import re
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from Bio import Phylo

# ==============================================================================
# 1. PARÁMETROS CONFIGURABLES
# ==============================================================================
ARCHIVO_INPUT = "PLA2_total3.NEXUS"

UMBRAL_MICRO_RAMA = 0.008
DPI_CALIDAD = 600
NOMBRE_SALIDA = "Arbol6_Filogenetico_Topologia_Nativa.png"
VALOR_BARRA_ESCALA = 0.2

# ==============================================================================
# 2. FUNCIONES DE PROCESAMIENTO Y RENDERIZADO CARTESIANO
# ==============================================================================

def procesar_arbol_nativo(ruta):
    """Detecta el formato real y respeta rigurosamente la raíz y topología de origen."""

    with open(ruta, 'r') as f:
        primera_linea = f.readline().strip().upper()

    formato_real = "nexus" if primera_linea.startswith("#NEXUS") else "newick"
    print(f"[INFO] Formato detectado: {formato_real.upper()}")

    try:
        arboles = list(Phylo.parse(ruta, formato_real))
        if not arboles:
            raise ValueError("El archivo se leyó pero no contiene árboles válidos.")
        arbol = arboles[0]
        print(f"[OK] Árbol cargado exitosamente conservando su topología y raíz nativa.")
    except Exception as e:
        raise RuntimeError(f"Error crítico al leer el archivo: {e}")

    # ladderize() ordena visualmente las ramas (clados pesados arriba/abajo)
    # pero NO altera la posición de la raíz evolutiva.
    arbol.ladderize()
    return arbol


def calcular_coordenadas_estiradas(arbol):
    """Calcula X e Y estirando el lienzo vertical para dar carril propio a cada rama."""
    pos_x = arbol.depths()
    terminales = arbol.get_terminals()

    pos_y = {term: float(idx * 1.38) for idx, term in enumerate(terminales)}

    for clado in arbol.find_clades(order='postorder'):
        if not clado.is_terminal():
            hijos_y = [pos_y[h] for h in clado.clades if h in pos_y]
            pos_y[clado] = (min(hijos_y) + max(hijos_y)) / 2.0 if hijos_y else 0.0

    return pos_x, pos_y


def renderizar_arbol_maestro(arbol, pos_x, pos_y):
    """Renderiza el árbol preservando colores y lógica anti-superposición."""
    num_hojas = len(arbol.get_terminals())
    alto_canvas = max(8.5, num_hojas * 0.44)
    ancho_canvas = max(10, alto_canvas * 0.82)

    fig, ax = plt.subplots(figsize=(ancho_canvas, alto_canvas), dpi=DPI_CALIDAD)
    max_x = max(pos_x.values())

    # A. DIBUJAR ESQUELETO DEL ÁRBOL
    for clado in arbol.find_clades():
        if clado in pos_x and clado in pos_y:
            x_p = pos_x[clado]
            if clado.clades:
                ys = [pos_y[h] for h in clado.clades if h in pos_y]
                if ys:
                    ax.plot([x_p, x_p], [min(ys), max(ys)], color='#1A1A1A', lw=1.1, zorder=1)
            for h in clado.clades:
                if h in pos_x and h in pos_y:
                    ax.plot([x_p, pos_x[h]], [pos_y[h], pos_y[h]], color='#1A1A1A', lw=1.1, zorder=1)

    raiz = arbol.root
    ax.plot([pos_x[raiz] - (max_x * 0.015), pos_x[raiz]], [pos_y[raiz], pos_y[raiz]], color='#1A1A1A', lw=1.1)

    # B. DIBUJAR NOMBRES DE ESPECIES
    for clado in arbol.get_terminals():
        nombre_limpio = clado.name.replace("'", "").replace("_", " ")
        ax.text(
            pos_x[clado] + (max_x * 0.008), pos_y[clado],
            nombre_limpio,
            va='center', ha='left',
            fontsize=10.5, fontfamily='serif', fontstyle='italic', color='#000000'
        )

    # C. BOOTSTRAPS (Soporte Universal)
    halo_blanco = [PathEffects.withStroke(linewidth=2.8, foreground="white")]

    for clado in arbol.find_clades():
        if not clado.is_terminal() and clado != arbol.root:
            longitud = clado.branch_length if clado.branch_length else 0.0
            soporte_num = None
            soporte_str = ""

            if clado.confidence is not None:
                soporte_num = float(clado.confidence)
            elif hasattr(clado, 'comment') and clado.comment:
                match = re.search(r'BS=([0-9.]+)', clado.comment)
                if match:
                    soporte_num = float(match.group(1))
            elif clado.name:
                try:
                    soporte_num = float(clado.name)
                except ValueError:
                    soporte_str = clado.name

            if soporte_num is not None:
                soporte_str = f"{int(soporte_num)}" if soporte_num.is_integer() else f"{soporte_num}"

            if soporte_str:
                padres = [p for p in arbol.find_clades() if clado in p.clades]
                x_padre = pos_x[padres[0]] if padres else pos_x[clado] - longitud

                es_alto_soporte = (soporte_num is not None and soporte_num >= 90.0)
                color_texto = "#C0392B" if es_alto_soporte else "#1A1A1A"
                peso_fuente = 'semibold' if es_alto_soporte else 'normal'

                if longitud >= UMBRAL_MICRO_RAMA:
                    x_opt = x_padre + (longitud * 0.40)
                    y_opt = pos_y[clado] - 0.22
                    ha_align = 'center'
                else:
                    x_opt = x_padre - (max_x * 0.005)
                    y_opt = pos_y[clado] + 0.18
                    ha_align = 'right'

                txt = ax.text(
                    x_opt, y_opt,
                    soporte_str,
                    va='center' if longitud < UMBRAL_MICRO_RAMA else 'top',
                    ha=ha_align,
                    fontsize=8.5, fontfamily='serif',
                    fontweight=peso_fuente, color=color_texto, zorder=4
                )
                txt.set_path_effects(halo_blanco)

    # D. BARRA DE ESCALA
    ax.axis('off')
    x_esc, y_esc = 0, -2.2

    ax.plot([x_esc, x_esc + VALOR_BARRA_ESCALA], [y_esc, y_esc], color='#1A1A1A', lw=1.4)
    ax.text(x_esc + (VALOR_BARRA_ESCALA / 2), y_esc - 0.35, f"{VALOR_BARRA_ESCALA}",
            va='top', ha='center', fontsize=9.5, fontfamily='serif')

    ax.set_xlim(- (max_x * 0.04), max_x * 1.38)
    ax.set_ylim(y_esc - 1.0, (num_hojas * 1.38) + 0.5)

    plt.savefig(NOMBRE_SALIDA, dpi=DPI_CALIDAD, bbox_inches='tight')
    plt.show()
    print(f"\n[ÉXITO TOTAL] Exporté la figura conservando la topología idéntica: '{NOMBRE_SALIDA}'.")

# ==============================================================================
# 3. EJECUCIÓN
# ==============================================================================
arbol_listo = procesar_arbol_nativo(ARCHIVO_INPUT)
px, py = calcular_coordenadas_estiradas(arbol_listo)
renderizar_arbol_maestro(arbol_listo, px, py)

[INFO] Formato detectado: NEXUS
[OK] Árbol cargado exitosamente conservando su topología y raíz nativa.



[ÉXITO TOTAL] Exporté la figura conservando la topología idéntica: 'Arbol6_Filogenetico_Topologia_Nativa.png'.
